In [4]:
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("tmdb_ds.csv")

df["title"] = df["title"].fillna("")

stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["clean_text"] = df["title"].apply(clean_text)


In [5]:

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (4803, 5000)


In [6]:
similarity_matrix = cosine_similarity(tfidf_matrix)
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (4803, 4803)


In [7]:
print("Similarity scores for the first five movies:")
print(similarity_matrix[:5, :5])

Similarity scores for the first five movies:
[[0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]


In [8]:
title_column = "title"  
print("First five movie titles:")
print(df[title_column].head())

print("\nSimilarity scores for the first movie:")
print(
    pd.DataFrame({
        "Movie": df[title_column],
        "Similarity Score": similarity_matrix[0]
    }).sort_values(
        by="Similarity Score",
        ascending=False
    ).head(6)
)

First five movie titles:
0                                      Avatar
1    Pirates of the Caribbean: At World's End
2                                     Spectre
3                       The Dark Knight Rises
4                                 John Carter
Name: title, dtype: str

Similarity scores for the first movie:
                                      Movie  Similarity Score
0                                    Avatar               0.0
1  Pirates of the Caribbean: At World's End               0.0
2                                   Spectre               0.0
3                     The Dark Knight Rises               0.0
4                               John Carter               0.0
5                              Spider-Man 3               0.0


Cosine similarity measures the angle between two numerical vectors. It is commonly used for text because it compares the direction of vectors rather than simply comparing their magnitudes.
- A score near 1 indicates highly similar text vectors.
- A score near 0 indicates little or no similarity.
- A score near -1 indicates opposite directions in general, although with standard non-negative TF-IDF vectors, scores are normally between 0 and 1.
For movie recommendations, cosine similarity helps identify movies whose descriptions contain similar words and phrases.